# 08 · Crossings and labels

Turns the flights into the labels the thermal and landout models learn from. Needs only the flights (05) and the terrain (06); weather (07) and terrain winds are joined later. Adds to `database/`:

| Table | One row per |
|---|---|
| `flight_checks` | flight: height above ground (median, share near the ground) and whether it is really a drive down from launch |
| `crossings` | flight and 1 km cell crossed: when, how long, height above ground in and out, time circling and climbing, best climb |
| `thermals` | thermal: base and top above the ground, the wind the pilot measured from the drift of the circles, and the trigger point traced back upwind |
| `landings` | flight: landing cell, time and height above ground |

Copies of another file's flight are left out. Use a High-RAM runtime with 8 CPUs. Tasks are cached in batches, so **if Colab disconnects, Run All again**.

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
WORKERS = 8          # processes (High-RAM runtime)
REDO = False         # True to rebuild every task (after a change to the method); False resumes
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
REPO = "/content/Pre-task-analyzer"

## 2. Mount Drive and get the code

In [ ]:
import os, shutil, subprocess
from google.colab import auth, drive
from googleapiclient.discovery import build

auth.authenticate_user()
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    """Folder names from a file up to the top of its drive, joined under root."""
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    """A writable local path for a Drive folder, from its id."""
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):  # in a shared drive
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):  # in my own Drive
        return path_up(folder_id, "/content/drive/MyDrive")
    # Shared with me: write through a shortcut to it in my Drive
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive "
                             "(right click -> Organize -> Add shortcut), then rerun this cell.")


ROOT = drive_path(DRIVE_FOLDER_ID)
DB = os.path.join(ROOT, "database")
DEM_DIR = os.path.join(ROOT, "dem", "glo30")
assert os.path.isfile(os.path.join(DB, "cells.parquet")), "Run notebooks 05 and 06 first"
subprocess.run(["pip", "install", "-q", "rasterio"], check=True)
print("Database in", DB)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)


def run(*cmd):
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()

## 3. Build the tables

Copies the DEM tiles to local disk, then works through the tasks. Prints progress every 100 tasks and a summary at the end.

In [ ]:
run(f"{REPO}/scripts/build_crossings.py", DB, "--dem", DEM_DIR, "--local", "/content/dem_glo30", "--workers", str(WORKERS), *(["--redo"] if REDO else []))

## 4. A first look

In [ ]:
import duckdb, pandas as pd
db = duckdb.connect()
for t in ["flight_checks", "crossings", "thermals", "landings", "results", "flights"]:
    db.execute(f"CREATE VIEW {t} AS SELECT * FROM read_parquet('{DB}/{t}.parquet')")

# Thermal base and top above the ground, by hour of the day (local solar time from longitude)
db.sql("""
    SELECT CAST(((t_start / 3600.0 + lon / 15.0 + 48) % 24) AS INT) AS solar_hour,
           count(*) AS thermals, round(median(agl_base)) AS base_m, round(median(agl_top)) AS top_m,
           round(median(rate_ms), 2) AS climb_ms
    FROM thermals WHERE agl_base IS NOT NULL
    GROUP BY 1 ORDER BY 1
""").df()

In [ ]:
# Wind measured by pilots, by height above ground: what WindNinja and the surrogate will be tested against (below 400 m)
db.sql("""
    SELECT CASE WHEN agl_base < 400 THEN '0-400 m' WHEN agl_base < 1000 THEN '400-1000 m' ELSE '1000+ m' END AS height,
           count(*) AS thermals, round(median(sqrt(wind_u*wind_u + wind_v*wind_v)), 1) AS wind_ms
    FROM thermals WHERE wind_u IS NOT NULL
    GROUP BY 1 ORDER BY 1
""").df()

In [ ]:
# Where pilots who did not reach goal landed, height above ground (close to 0 means the landing was found)
db.sql("""
    SELECT r.status, count(*) AS flights, round(median(l.agl)) AS landing_agl_m
    FROM landings l JOIN results r USING (result_id)
    WHERE NOT l.is_drive
    GROUP BY 1 ORDER BY 2 DESC
""").df()